# Car Price Prediction System

**SPPU PR12 practical ? Eight-input model**

This notebook documents the data, preprocessing, model selection, tuning, and final held-out evaluation used by the FastAPI service. It reads the supplied listing details and joins them to `listed_price` by `usedCarSkuId`.

## 1. Problem definition and objectives

Estimate a used car's advertised listing price in INR from exactly eight values: company, model, vehicle age, kilometres driven, mileage (km/l), seating capacity, horsepower (bhp), and engine displacement (cc). This is supervised regression. The result is an educational estimate, not a guaranteed transaction price.

## 2. Supplied data and schema checks

`cars_details_merges.csv` contains the vehicle specifications. `cars_data_clean.csv` contains the numeric `listed_price` target. The files are joined using the unique `usedCarSkuId`. The feature dictionary names the corresponding source columns. No replacement data is downloaded.

In [ ]:
import pandas as pd
from backend.train_model import (
    DATA_DETAILS, DATA_TARGET, FEATURES, TARGET, REFERENCE_YEAR,
    prepare_training_data,
)
details = pd.read_csv(DATA_DETAILS, nrows=5)
target = pd.read_csv(DATA_TARGET, nrows=5)
print("Details columns:", len(pd.read_csv(DATA_DETAILS, nrows=0).columns))
print("Target columns:", pd.read_csv(DATA_TARGET, nrows=0).columns.tolist())
print("Detail sample columns:", details.columns.tolist())
print("Target sample:"); display(target.head())
print("Feature dictionary:")
display(pd.read_csv("Dataset/feature_dictionary.csv").query("Feature in ['myear','km','Max Power','Displacement','Seating Capacity','mileage_new','brand_name','model_name']"))

## 3. Data preparation and unit handling

The source has `myear`, not age, so age is derived as **2026 ? manufacturing year**. Mileage is kept only when explicitly recorded in kmpl; km/kg and missing mileage are excluded. Power is converted from bhp, hp, PS, or kW into bhp. Missing/zero engine displacement is excluded because cc is not meaningful for electric vehicles. One malformed 48 cc e2o listing is also excluded; 100 cc is the documented lower sanity bound. One implausible target listing above ?100 million is excluded; other premium prices are retained. The exact counts are reported by the training function.

In [ ]:
X, y, audit = prepare_training_data()
print("Usable shape:", X.shape)
print("Input order:", FEATURES)
print("Audit:", audit)
display(X.describe(include="all").T)
display(y.describe(percentiles=[.01,.5,.9,.99]))

## 4. Exploratory analysis

Plots show the observed target and selected numerical relationships. The right-hand price view omits only values above the documented quality threshold so the central distribution remains visible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(y, bins=60, ax=axes[0]); axes[0].set_title("Listed price (INR)")
sns.histplot(y[y <= 10_000_000], bins=50, ax=axes[1]); axes[1].set_title("Listed price through ?10 million")
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.scatterplot(x=X.vehicle_age, y=y, alpha=.2, s=12, ax=axes[0])
axes[0].set(title="Price by vehicle age", xlabel="Age (years; 2026 reference)", ylabel="Price (INR)")
sns.scatterplot(x=X.km_driven, y=y, alpha=.2, s=12, ax=axes[1])
axes[1].set_xlim(0, X.km_driven.quantile(.99)); axes[1].set(title="Price by kilometres driven", xlabel="Kilometres", ylabel="Price (INR)")
plt.tight_layout()

## 5. Training split and model comparison

A reproducible 80/20 split is used. Preprocessing is inside each scikit-learn pipeline, so imputation, scaling, and category encoding are fit only on training folds. The comparison includes a median baseline, Linear Regression, Decision Tree, Random Forest, and HistGradientBoosting.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.20, random_state=42
)
print("Training rows:", len(X_train), "Test rows:", len(X_test))

## 6. Hyperparameter tuning and final evaluation

The selected candidate is tuned using randomized search and 3-fold cross-validation on training data only. The test set is used once for final MAE, RMSE, MSE, and R?.

In [ ]:
from backend.train_model import main
comparison, final_metrics = main()
display(comparison)
print("Held-out test metrics:", final_metrics)

In [ ]:
comparison = pd.read_csv("models/model_comparison.csv")
plot_data = comparison.dropna(subset=["CV_MAE_INR"]).sort_values("CV_MAE_INR")
plt.figure(figsize=(10, 4))
sns.barplot(data=plot_data, x="CV_MAE_INR", y="Model", color="#3b82f6")
plt.xlabel("3-fold CV MAE (INR; lower is better)"); plt.ylabel("")
plt.title("Model comparison on training folds"); plt.tight_layout()

## 7. Residual analysis and saved pipeline check

MAE is the typical absolute miss in rupees. RMSE is in rupees and penalizes large errors more heavily. MSE is in rupees squared. R? summarizes explained variation against a mean baseline. These metrics describe this dataset split; they do not guarantee accuracy on newer cars or different market segments.

In [ ]:
import joblib
results = pd.read_csv("models/test_predictions.csv")
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(results.actual_price, results.predicted_price, alpha=.3, s=12)
upper = max(results.actual_price.max(), results.predicted_price.max())
axes[0].plot([0, upper], [0, upper], "r--")
axes[0].set(xlabel="Actual (INR)", ylabel="Predicted (INR)", title="Actual vs predicted")
residual = results.actual_price - results.predicted_price
axes[1].scatter(results.predicted_price, residual, alpha=.3, s=12)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set(xlabel="Predicted (INR)", ylabel="Residual (INR)", title="Residuals")
plt.tight_layout()
model = joblib.load("models/car_price_model.joblib")
print("Reloaded pipeline:", type(model).__name__, "input features:", model.feature_names_in_.tolist())

## 8. Conclusion and limitations

Use the actual printed comparison and test metrics when presenting this practical; do not report regression scores as an accuracy percentage. The eight fields exclude condition, exact trim, city, fuel, transmission, owner history, service history, and sale timing. Mileage rows without kmpl units and electric vehicles without engine cc are not represented in this version. The estimate is for advertised listings, not a verified selling price.